In [ ]:
%%spark

# ============================================================
# 3_PREPARACAO_CONTROLE
# ============================================================

def _ler_ctl_oficial(database: str):
    tabela_ctl = TABELAS_HIVE["hive_controle_operacional_recomendacao"]
    return ler_tabela_hive(database, tabela_ctl)


def _carregar_ctl_oficial_validado(database: str, tabela_ctl: str, logger_etapa):
    objeto = nome_tabela_hive(database, tabela_ctl)

    try:
        df_lido = _ler_ctl_oficial(database)
        validar_estrutura_ctl(
            df_lido,
            "CTL_OPRL_RCM_OFICIAL",
            logger_etapa=logger_etapa,
        )
        df_validado = projetar_contrato_hive(df_lido, tabela_ctl)
        ctl_vazio = not dataframe_tem_registros(df_validado)
        return df_validado, ctl_vazio
    except ErroPipeline as exc:
        if not exc.etapa:
            exc.etapa = "PREPARACAO_CONTROLE"
        if not exc.objeto:
            exc.objeto = objeto
        raise
    except Exception as exc:
        raise ErroOperacional(
            codigo="CTL_OFICIAL_INVALIDO",
            mensagem="Falha inesperada ao preparar a tabela de controle oficial.",
            etapa="PREPARACAO_CONTROLE",
            objeto=objeto,
            acao="Verificar a causa raiz e a consistencia da tabela de controle.",
        ) from exc


def _validar_backup_ctl(path_temp_backup: str, logger_etapa):
    if not path_hdfs_existe(path_temp_backup):
        return None

    try:
        df_backup = ler_parquet_sql(path_temp_backup)
        validar_estrutura_ctl(
            df_backup,
            "TEMP_BACKUP_CTL_OPRL_RCM",
            logger_etapa=logger_etapa,
        )
        df_backup = projetar_contrato_hive(
            df_backup,
            TABELAS_HIVE["hive_controle_operacional_recomendacao"],
        )
    except Exception as exc:
        raise ErroOperacional(
            codigo="CTL_BACKUP_INVALIDO",
            mensagem="Foto segura da tabela de controle esta invalida ou inacessivel.",
            etapa="PREPARACAO_CONTROLE",
            objeto=path_temp_backup,
            acao="Analisar a foto segura antes de decidir pela recuperacao ou primeira carga.",
        ) from exc

    if not dataframe_tem_registros(df_backup):
        raise ErroOperacional(
            codigo="CTL_BACKUP_INVALIDO",
            mensagem="Foto segura da tabela de controle esta vazia.",
            etapa="PREPARACAO_CONTROLE",
            objeto=path_temp_backup,
            detalhes={"motivo": "backup_vazio"},
            acao="Analisar a recuperacao operacional antes de reiniciar o fluxo.",
        )

    return df_backup


def executar_preparacao_controle(
    database: str,
    path_save_hdfs: str,
    primeira_carga: bool,
    executar: bool,
    hoje: str,
) -> dict:
    logger_etapa = logger
    tabela_ctl = TABELAS_HIVE["hive_controle_operacional_recomendacao"]
    caminhos = caminhos_temporarios_ctl(path_save_hdfs)

    stats = {
        "primeira_carga": primeira_carga,
        "dry_run": not executar,
        "backup_existente": False,
        "backup_usado": False,
        "ctl_vazio": None,
    }

    logger_etapa.info(
        f"[PREPARACAO_CONTROLE][INICIO] Iniciando preparacao do CTL_OPRL_RCM. executar={executar} primeira_carga={primeira_carga}",
    )

    if primeira_carga:
        df_ctl_oficial, ctl_vazio = _carregar_ctl_oficial_validado(
            database,
            tabela_ctl,
            logger_etapa,
        )
        stats["ctl_vazio"] = ctl_vazio

        if not ctl_vazio:
            raise ErroOperacional(
                codigo="PRIMEIRA_CARGA_CTL_NAO_VAZIA",
                mensagem="Primeira carga exige a tabela de controle oficial vazia.",
                etapa="PREPARACAO_CONTROLE",
                objeto=nome_tabela_hive(database, tabela_ctl),
                acao="Esvaziar a CTL somente se a primeira carga estiver operacionalmente autorizada; caso contrario, executar como carga normal.",
            )

        if executar:
            try:
                backup_removido = remover_path_hdfs_se_existir(caminhos["temp_backup"], executar=True, logger_etapa=logger_etapa, contexto="PRIMEIRA_CARGA")
                lineage_removido = remover_path_hdfs_se_existir(caminhos["temp_lineage"], executar=True, logger_etapa=logger_etapa, contexto="PRIMEIRA_CARGA")
            except ErroPipeline:
                raise
            except Exception as exc:
                raise ErroOperacional(
                    codigo="HDFS_LIMPEZA_TEMPORARIOS_FALHOU",
                    mensagem="Falha ao limpar temporarios HDFS da primeira carga.",
                    etapa="PREPARACAO_CONTROLE",
                    objeto=path_save_hdfs,
                    detalhes={"paths": list(caminhos.values())},
                    acao="Verificar existencia e permissao dos paths temporarios HDFS.",
                ) from exc

            if not backup_removido or not lineage_removido:
                raise ErroOperacional(
                    codigo="HDFS_LIMPEZA_TEMPORARIOS_FALHOU",
                    mensagem="Limpeza dos temporarios HDFS da primeira carga nao foi concluida.",
                    etapa="PREPARACAO_CONTROLE",
                    objeto=path_save_hdfs,
                    detalhes={
                        "temp_backup_removido": backup_removido,
                        "temp_lineage_removido": lineage_removido,
                    },
                    acao="Verificar os paths temporarios antes de reiniciar a primeira carga.",
                )

            logger_etapa.info("[PREPARACAO_CONTROLE][PRIMEIRA_CARGA] CTL vazia validada. A inicializacao pelo Oracle ocorrera na sincronizacao.")
        else:
            logger_etapa.info(
                "[PREPARACAO_CONTROLE][DRY_RUN] primeira_carga=True com CTL vazia. temp_backup e temp_lineage antigos seriam removidos se executar=True.",
            )

        return {
            "df_ctl_oprl_rcm": df_ctl_oficial,
            "stats": stats,
        }

    temp_backup_existe = path_hdfs_existe(caminhos["temp_backup"])
    stats["backup_existente"] = temp_backup_existe

    if temp_backup_existe:
        logger_etapa.info("[RECUPERACAO][TEMP_BACKUP_DETECTADO] backup existente; recuperacao automatica iniciada.")
        try:
            df_backup = _validar_backup_ctl(caminhos["temp_backup"], logger_etapa)
            if df_backup is None:
                raise ErroOperacional(
                    codigo="CTL_BACKUP_INVALIDO",
                    mensagem="Foto segura desapareceu durante a recuperacao automatica.",
                    etapa="PREPARACAO_CONTROLE",
                    objeto=caminhos["temp_backup"],
                    acao="Verificar o path HDFS e reiniciar somente apos analise operacional.",
                )
        except Exception as exc:
            erro_backup = exc
            try:
                _, ctl_diagnostico_vazio = _carregar_ctl_oficial_validado(
                    database,
                    tabela_ctl,
                    logger_etapa,
                )
            except Exception as diag_exc:
                raise ErroOperacional(
                    codigo="CTL_BACKUP_INVALIDO",
                    mensagem="Foto segura invalida e tabela oficial indisponivel para diagnostico.",
                    etapa="PREPARACAO_CONTROLE",
                    objeto=caminhos["temp_backup"],
                    detalhes={
                        "diagnostico_ctl": getattr(diag_exc, "codigo", type(diag_exc).__name__),
                    },
                    acao="Analisar a foto segura e a tabela oficial antes de nova execucao.",
                ) from erro_backup

            stats["ctl_vazio"] = ctl_diagnostico_vazio

            if ctl_diagnostico_vazio is True:
                raise ErroOperacional(
                    codigo="CTL_SEM_ESTADO_RECUPERAVEL",
                    mensagem="Foto segura invalida e tabela de controle oficial vazia; nao existe estado recuperavel.",
                    etapa="PREPARACAO_CONTROLE",
                    objeto=nome_tabela_hive(database, tabela_ctl),
                    detalhes={"temp_backup": caminhos["temp_backup"]},
                    acao="Executar primeira_carga=True somente se a perda do historico operacional estiver autorizada.",
                ) from erro_backup

            raise ErroOperacional(
                codigo="CTL_BACKUP_INVALIDO",
                mensagem="Foto segura invalida; recuperacao automatica foi bloqueada.",
                etapa="PREPARACAO_CONTROLE",
                objeto=caminhos["temp_backup"],
                acao="Analisar a foto segura; a tabela oficial nao sera usada automaticamente como fallback.",
            ) from erro_backup

        stats["backup_usado"] = True
        logger_etapa.info("[RECUPERACAO][TEMP_BACKUP_USADO] backup validado e usado como fonte da verdade.")

        return {
            "df_ctl_oprl_rcm": df_backup,
            "stats": stats,
        }

    df_ctl_oficial, ctl_vazio = _carregar_ctl_oficial_validado(
        database,
        tabela_ctl,
        logger_etapa,
    )
    stats["ctl_vazio"] = ctl_vazio

    if not ctl_vazio:
        if executar:
            # EXCECAO_SQL: escrita Parquet temporaria e efeito fisico de recuperacao operacional.
            try:
                df_ctl_oficial.write.mode("overwrite").parquet(caminhos["temp_backup"])
            except Exception as exc:
                raise ErroOperacional(
                    codigo="CTL_BACKUP_CRIACAO_FALHOU",
                    mensagem="Falha ao criar a foto segura da tabela de controle.",
                    etapa="PREPARACAO_CONTROLE",
                    objeto=caminhos["temp_backup"],
                    acao="Verificar permissao e disponibilidade do path HDFS de recuperacao.",
                ) from exc
            logger_etapa.info(f"[PREPARACAO_CONTROLE][TEMP_BACKUP_CRIADO] nova foto segura criada no fluxo normal. path={caminhos['temp_backup']}")
        else:
            logger_etapa.info(
                f"[PREPARACAO_CONTROLE][DRY_RUN] temp_backup seria criado se executar=True. path={caminhos['temp_backup']}",
            )

        return {
            "df_ctl_oprl_rcm": df_ctl_oficial,
            "stats": stats,
        }

    raise ErroOperacional(
        codigo="CTL_SEM_ESTADO_RECUPERAVEL",
        mensagem="Execucao normal encontrou a tabela de controle vazia e nenhum backup valido.",
        etapa="PREPARACAO_CONTROLE",
        objeto=nome_tabela_hive(database, tabela_ctl),
        acao="Executar primeira_carga=True somente quando o reinicio sem historico estiver autorizado.",
    )
